# PyTorch Tensor Exercises

*ML & NLP course — Data Trainers LLC — Axel Sirota*

## The story

You're a data scientist who has been working with TensorFlow for years. Your new team runs everything on **PyTorch** — models, pipelines, training loops, the works. Before you can contribute to any real model, you need the same fluency you have in TF, but expressed in PyTorch idioms.

This notebook is your on-ramp. You'll work through the eight tensor-manipulation topics that underpin **every model in this course** — from logistic regression all the way to LSTM text generation. Think of these as Lego bricks: once the individual pieces are in your hands, snapping them together into architectures becomes mechanical.

## Learning objectives

By the end of this notebook you will be able to:

1. Create tensors of any shape and dtype with `torch.tensor`, `torch.zeros/ones/rand/randn`.
2. Perform arithmetic, statistical, and matrix operations on tensors.
3. Reshape tensors with `.reshape`, `.view`, `.transpose`, `.permute`, `.flatten`, `.squeeze`/`.unsqueeze`.
4. Index and slice tensors, including boolean masking.
5. Explain broadcasting rules and apply them confidently.
6. Compute gradients with `requires_grad=True` and `loss.backward()` — the PyTorch replacement for `tf.GradientTape`.
7. Build a `TensorDataset` + `DataLoader` pipeline — the PyTorch replacement for `tf.data`.
8. Assemble a forward pass with `nn.Linear`, `nn.ReLU`, a loss function, and backprop.

## Prerequisites

- Python 3.8+, basic NumPy familiarity.
- No prior PyTorch knowledge required — that's the point.

## How to run

- **Colab (recommended):** Runtime → Change runtime type → GPU.
- **CPU:** Every exercise in this notebook runs fine on CPU — tensors are small.

Let's build it.

## Section 0 — Environment Setup

Install PyTorch and NumPy (Colab already has them; this cell is a safety net), then import everything, set seeds, and detect the device.

In [ ]:
# Install required packages (run this first on Google Colab).
# If running locally in a virtualenv that already has these, you can skip.
!pip install -q torch numpy

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ---- Reproducibility ----
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ---- Device detection (GPU if available, otherwise CPU) ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version : {torch.__version__}")
print(f"NumPy version   : {np.__version__}")
print(f"Using device    : {device}")
if device.type == 'cuda':
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")

print("\nEnvironment setup complete.")

## Section 1 — Tensor Basics

**The mental model:** a PyTorch tensor is NumPy array + optional GPU placement + optional autograd tracking. The factory functions are almost a 1-to-1 mapping:

| NumPy | PyTorch |
|-------|---------|
| `np.array([1, 2, 3])` | `torch.tensor([1, 2, 3])` |
| `np.zeros((3, 3))` | `torch.zeros(3, 3)` |
| `np.ones((2, 4))` | `torch.ones(2, 4)` |
| `np.random.rand(2, 3)` | `torch.rand(2, 3)` |
| `np.random.randn(2, 3)` | `torch.randn(2, 3)` |

Two attributes you'll check constantly:

```python
t = torch.tensor([1.0, 2.0, 3.0])
t.shape   # torch.Size([3])
t.dtype   # torch.float32
```

**Numpy ↔ PyTorch conversion:**

```python
arr = np.array([1.0, 2.0])
t   = torch.from_numpy(arr)     # shares memory — mutating arr changes t
t2  = torch.tensor(arr)         # copies — independent
back = t.numpy()                # back to numpy (CPU tensors only)
```

### Demo — factory functions in action

In [ ]:
# Demo: tensor creation and inspection
scalar  = torch.tensor(3.14)                       # 0-D tensor
vector  = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0]) # 1-D
matrix  = torch.zeros(3, 3)                        # 2-D all zeros
rand3d  = torch.randn(2, 3, 4)                     # 3-D standard normal

for name, t in [('scalar', scalar), ('vector', vector),
                ('matrix', matrix), ('rand3d', rand3d)]:
    print(f"{name:8s}  shape={str(t.shape):20s}  dtype={t.dtype}")

# NumPy round-trip
arr  = np.array([[1.0, 2.0], [3.0, 4.0]])
t_np = torch.from_numpy(arr)     # zero-copy view
back = t_np.numpy()              # back to numpy
print(f"\nNumPy -> torch -> NumPy: arr type={type(arr).__name__}, "
      f"t_np type={type(t_np).__name__}, back type={type(back).__name__}")
print(f"All values identical: {np.allclose(arr, back)}")

### Lab 1 — Tensor Basics

Complete the tasks below. Verification code is provided so you can check your work immediately.

**Tasks:**
1. Create a scalar tensor with the value `7.0` and name it `s`.
2. Create a 1-D tensor with exactly 5 elements `[10, 20, 30, 40, 50]` as `torch.float32` and name it `v`.
3. Create a 2-D tensor of shape `(3, 3)` filled with ones and name it `m`.
4. Create a 3-D tensor of shape `(2, 4, 4)` filled with random normal values and name it `t3d`.
5. Convert the NumPy array `np_arr` below to a PyTorch tensor named `t_from_np`.
6. Convert `t_from_np` back to a NumPy array named `arr_back`.

In [ ]:
np_arr = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

# 1. Scalar tensor (value 7.0)
s = None  # YOUR CODE

# 2. 1-D tensor [10, 20, 30, 40, 50] as float32
v = None  # YOUR CODE

# 3. 2-D tensor of ones, shape (3, 3)
m = None  # YOUR CODE

# 4. 3-D tensor of random normal values, shape (2, 4, 4)
t3d = None  # YOUR CODE

# 5. Convert np_arr to a PyTorch tensor
t_from_np = None  # YOUR CODE

# 6. Convert t_from_np back to a NumPy array
arr_back = None  # YOUR CODE

# ---- Verification ----
if s is not None:
    print(f"s          : value={s.item():.1f}  shape={s.shape}  dtype={s.dtype}")
if v is not None:
    print(f"v          : shape={v.shape}  dtype={v.dtype}  values={v.tolist()}")
if m is not None:
    print(f"m          : shape={m.shape}  all_ones={m.all().item()}")
if t3d is not None:
    print(f"t3d        : shape={t3d.shape}  dtype={t3d.dtype}")
if t_from_np is not None:
    print(f"t_from_np  : shape={t_from_np.shape}  dtype={t_from_np.dtype}")
if arr_back is not None:
    print(f"arr_back   : type={type(arr_back).__name__}  shape={arr_back.shape}")

## Section 2 — Tensor Operations

PyTorch operators map almost identically to NumPy — the key difference is that they work on the GPU and track gradients.

**Arithmetic:** `+`, `-`, `*`, `/` work element-wise. Alternatively `torch.add`, `torch.sub`, `torch.mul`, `torch.div`.

**Statistics:**
```python
t.mean()          # scalar mean over all elements
t.mean(dim=0)     # mean along dim 0 (collapse rows -> one value per column)
t.std()           # standard deviation
```

**Activation functions:**
```python
torch.relu(t)     # max(0, x) element-wise — no import needed
nn.functional.relu(t)  # same, via nn module
```

**Matrix multiply:**
```python
A @ B             # equivalent to torch.matmul(A, B)
torch.mm(A, B)    # strict 2-D only version
```

### Demo — operations

In [ ]:
torch.manual_seed(SEED)

a = torch.tensor([1.0, 2.0, 3.0, 4.0])
b = torch.tensor([10.0, 20.0, 30.0, 40.0])

print("a + b  =", (a + b).tolist())
print("a - b  =", (a - b).tolist())
print("a * b  =", (a * b).tolist())
print("a / b  =", (a / b).tolist())

print(f"\na.mean() = {a.mean().item():.4f}")
print(f"a.std()  = {a.std().item():.4f}")

# ReLU applied to a tensor with negative values
c = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
print(f"\ntorch.relu({c.tolist()}) = {torch.relu(c).tolist()}")

# Matrix multiply: (2,3) @ (3,2) -> (2,2)
A = torch.ones(2, 3)
B = torch.ones(3, 2) * 2.0
C = A @ B
print(f"\n(2x3) @ (3x2) = {C.shape}, values:\n{C}")

### Lab 2 — Tensor Operations

**Tasks:**
1. Create tensors `x = [1.0, 4.0, 9.0, 16.0]` and `y = [2.0, 2.0, 3.0, 4.0]`. Compute element-wise add, subtract, multiply, and divide; store in `add_xy`, `sub_xy`, `mul_xy`, `div_xy`.
2. Create a `(4, 4)` random normal tensor `mat`. Compute its global mean (`mat_mean`) and standard deviation (`mat_std`).
3. Apply ReLU to `torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])` and store the result in `relu_out`.
4. Create matrices `P = torch.ones(3, 4)` and `Q = torch.ones(4, 5) * 3.0`. Compute `PQ = P @ Q`.

In [ ]:
torch.manual_seed(SEED)

# 1. Element-wise arithmetic
x = torch.tensor([1.0, 4.0, 9.0, 16.0])
y = torch.tensor([2.0, 2.0, 3.0,  4.0])

add_xy = None  # YOUR CODE
sub_xy = None  # YOUR CODE
mul_xy = None  # YOUR CODE
div_xy = None  # YOUR CODE

# 2. Mean and std of a (4,4) random matrix
mat = torch.randn(4, 4)
mat_mean = None  # YOUR CODE
mat_std  = None  # YOUR CODE

# 3. ReLU
relu_out = None  # YOUR CODE  (apply to torch.tensor([-2., -1., 0., 1., 2.]))

# 4. Matrix multiply
P  = torch.ones(3, 4)
Q  = torch.ones(4, 5) * 3.0
PQ = None  # YOUR CODE

# ---- Verification ----
if add_xy is not None:
    print(f"x + y  = {add_xy.tolist()}")
    print(f"x - y  = {sub_xy.tolist()}")
    print(f"x * y  = {mul_xy.tolist()}")
    print(f"x / y  = {div_xy.tolist()}")
if mat_mean is not None:
    print(f"\nmat mean = {mat_mean.item():.4f}  std = {mat_std.item():.4f}")
if relu_out is not None:
    print(f"relu_out = {relu_out.tolist()}")
if PQ is not None:
    print(f"PQ shape = {PQ.shape}  (expected (3, 5))  values all 12.0: {(PQ == 12.0).all().item()}")

## Section 3 — Manipulating Shapes

Reshaping is one of the most common operations you'll perform when wiring model layers together — a layer might need `(B, T, D)` while the next expects `(B, D*T)`.

**Key methods:**

| Method | Notes |
|--------|-------|
| `.reshape(new_shape)` | Returns a view if possible, otherwise a copy. Safe general purpose. |
| `.view(new_shape)` | Always returns a view — tensor **must** be contiguous in memory. Faster but stricter. |
| `.transpose(dim0, dim1)` | Swaps two dimensions. Returns a view. |
| `.permute(*dims)` | Reorders all dimensions at once. Returns a view. |
| `.flatten(start_dim, end_dim)` | Collapses a range of dims into one. |
| `.squeeze(dim)` | Removes dims of size 1. No `dim` = remove all size-1 dims. |
| `.unsqueeze(dim)` | Inserts a new size-1 dim at position `dim`. |

**Common pattern — adding a batch dimension:**
```python
x = torch.randn(28, 28)          # single image
x = x.unsqueeze(0)               # (1, 28, 28) — batch of 1
x = x.unsqueeze(0)               # (1, 1, 28, 28) — add channel dim too
```

### Demo — shape manipulation

In [ ]:
t = torch.arange(16, dtype=torch.float32)  # [0, 1, ..., 15]
print("Original shape:", t.shape)

r1 = t.reshape(4, 4)
print("reshape(4,4)   :", r1.shape)

r2 = t.reshape(2, 8)
print("reshape(2,8)   :", r2.shape)

# transpose swaps two dims
r3 = r1.transpose(0, 1)   # (4,4) -> (4,4) but rows and cols swapped
print("transpose(0,1) :", r3.shape, "  (rows become columns)")

# permute on a 3-D tensor
t3 = torch.randn(2, 3, 5)
r4 = t3.permute(2, 0, 1)   # (2,3,5) -> (5,2,3)
print(f"permute(2,0,1) on {t3.shape}: {r4.shape}")

# flatten and squeeze/unsqueeze
t4 = torch.zeros(2, 1, 3, 1)
print(f"\nsqueeze()      : {t4.shape} -> {t4.squeeze().shape}")
print(f"unsqueeze(0)   : {t.shape} -> {t.unsqueeze(0).shape}")
print(f"flatten()      : {r1.shape} -> {r1.flatten().shape}")

### Lab 3 — Manipulating Shapes

**Tasks:**
1. Create `base = torch.arange(24, dtype=torch.float32)`. Reshape it to `(3, 8)` → `reshaped_3x8`.
2. Reshape `base` to `(2, 3, 4)` → `reshaped_3d`.
3. Transpose `reshaped_3x8` (swap dim 0 and dim 1) → `transposed`.
4. Permute `reshaped_3d` from `(2, 3, 4)` to `(4, 2, 3)` → `permuted`.
5. Flatten `reshaped_3d` entirely → `flat`.
6. Start with `img = torch.randn(28, 28)`. Add a batch dimension at position 0 → `img_batched` (shape should be `(1, 28, 28)`).
7. Remove that batch dimension → `img_back` (shape should be `(28, 28)`).

In [ ]:
base = torch.arange(24, dtype=torch.float32)
img  = torch.randn(28, 28)

# 1. Reshape to (3, 8)
reshaped_3x8 = None  # YOUR CODE

# 2. Reshape to (2, 3, 4)
reshaped_3d  = None  # YOUR CODE

# 3. Transpose reshaped_3x8 (dim0 <-> dim1)
transposed   = None  # YOUR CODE

# 4. Permute reshaped_3d from (2,3,4) to (4,2,3)
permuted     = None  # YOUR CODE

# 5. Flatten reshaped_3d entirely
flat         = None  # YOUR CODE

# 6. Add batch dim to img -> (1, 28, 28)
img_batched  = None  # YOUR CODE

# 7. Remove that batch dim -> (28, 28)
img_back     = None  # YOUR CODE

# ---- Verification ----
checks = [
    ('reshaped_3x8', reshaped_3x8, (3, 8)),
    ('reshaped_3d',  reshaped_3d,  (2, 3, 4)),
    ('transposed',   transposed,   (8, 3)),
    ('permuted',     permuted,     (4, 2, 3)),
    ('flat',         flat,         (24,)),
    ('img_batched',  img_batched,  (1, 28, 28)),
    ('img_back',     img_back,     (28, 28)),
]
for name, t, expected in checks:
    if t is not None:
        status = "OK" if tuple(t.shape) == expected else f"WRONG (got {tuple(t.shape)})"
        print(f"{name:15s} shape={tuple(t.shape)}  expected={expected}  [{status}]")

## Section 4 — Indexing and Slicing

PyTorch indexing syntax is identical to NumPy — if you know one, you know the other.

**Basic indexing:**
```python
t = torch.tensor([[1, 2, 3],
                  [4, 5, 6],
                  [7, 8, 9]])

t[0]        # first row:  tensor([1, 2, 3])
t[0, 2]     # row 0, col 2:  tensor(3)
t[1:3, :]   # rows 1 and 2, all cols
t[:, 1]     # all rows, column 1
```

**Boolean masking** — select elements that satisfy a condition:
```python
t[t > 4]    # all elements greater than 4
```

This is extremely useful for filtering token embeddings, masking padding tokens, etc.

### Demo — indexing and slicing

In [ ]:
grid = torch.arange(1, 10, dtype=torch.float32).reshape(3, 3)
print("grid:\n", grid)

print("\ngrid[0]          =", grid[0].tolist(),        "  (first row)")
print("grid[0, 2]       =", grid[0, 2].item(),        "  (row 0, col 2)")
print("grid[1:3, :]     =\n", grid[1:3, :])
print("grid[:, 1]       =", grid[:, 1].tolist(),      "  (middle column)")

# Boolean masking
mask = grid > 4
print("\ngrid > 4 mask:\n", mask)
print("grid[grid > 4]   =", grid[grid > 4].tolist())  # elements > 4

### Lab 4 — Indexing and Slicing

Use the tensor `data` created in the starter code below.

**Tasks:**
1. Extract the element at row 1, column 2 → `elem`.
2. Extract rows 1 and 2 (all columns) → `rows_1_2`.
3. Extract the last column → `last_col`.
4. Use boolean masking to get all elements less than 0 → `negatives`.
5. Use boolean masking to get all elements between 3 and 7 inclusive → `mid_range`.

In [ ]:
torch.manual_seed(SEED)
data = torch.randn(4, 5) * 5   # (4, 5) random values in roughly [-15, 15]
print("data:\n", data.round(decimals=2))

# 1. Element at row 1, col 2
elem      = None  # YOUR CODE

# 2. Rows 1 and 2, all columns
rows_1_2  = None  # YOUR CODE

# 3. Last column (all rows)
last_col  = None  # YOUR CODE

# 4. All elements less than 0
negatives = None  # YOUR CODE

# 5. All elements between 3 and 7 inclusive
mid_range = None  # YOUR CODE

# ---- Verification ----
if elem is not None:
    print(f"\nelem       = {elem.item():.4f}")
if rows_1_2 is not None:
    print(f"rows_1_2   shape={rows_1_2.shape}  (expected (2, 5))")
if last_col is not None:
    print(f"last_col   shape={last_col.shape}  (expected (4,))")
if negatives is not None:
    print(f"negatives  = {negatives.round(decimals=2).tolist()}")
if mid_range is not None:
    print(f"mid_range  = {mid_range.round(decimals=2).tolist()}")

## Section 5 — Broadcasting

Broadcasting lets PyTorch operate on tensors of *different* shapes without copying data. The rules (same as NumPy):

1. If the tensors have different numbers of dimensions, **prepend 1s** to the smaller shape.
2. Dimensions of size **1 are stretched** to match the other tensor.
3. If sizes still don't match after rule 1 and 2 → **error**.

**Shape arithmetic example:**

```
a shape:     (3, 4)
b shape:        (4,)   ->  prepend 1  ->  (1, 4)  ->  stretch  ->  (3, 4)
result:      (3, 4)   ✓

a shape:     (3, 1)
b shape:     (1, 4)
result:      (3, 4)   ✓  — both dimensions are stretched

a shape:     (3, 4)
b shape:     (3, 3)
result:      ERROR    ✗  — 4 and 3 can't be broadcast
```

Broadcasting is how bias addition works in `nn.Linear`: bias is `(out_features,)` but gets added to each row of `(batch_size, out_features)`.

### Demo — broadcasting

In [ ]:
# Case 1: (3, 4) + (4,)  ->  (3, 4)
M = torch.ones(3, 4)          # each row is [1,1,1,1]
row = torch.tensor([0.0, 1.0, 2.0, 3.0])  # shape (4,)
result1 = M + row
print("(3,4) + (4,) ->", result1.shape)
print(result1)

# Case 2: (3, 1) + (1, 4)  ->  (3, 4)
col = torch.tensor([[10.0], [20.0], [30.0]])   # (3, 1)
row2 = torch.tensor([[1.0, 2.0, 3.0, 4.0]])   # (1, 4)
result2 = col + row2
print("\n(3,1) + (1,4) ->", result2.shape)
print(result2)

# Case 3: intentional failure to show the error message
try:
    bad = torch.ones(3, 4) + torch.ones(3, 3)
except RuntimeError as e:
    print(f"\nExpected error: {e}")

### Lab 5 — Broadcasting

**Tasks:**
1. Add a bias vector `bias = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])` to every row of `features` (shape `(6, 5)`). Store the result in `biased`. Verify shape is `(6, 5)`.
2. Multiply a column scaling vector `col_scale = torch.tensor([[2.0], [3.0], [4.0]])` (shape `(3, 1)`) by a row weight vector `row_weights = torch.tensor([[1.0, 0.5, 0.25]])` (shape `(1, 3)`). Store result in `scaled`. Verify shape is `(3, 3)` and describe in a comment what each entry means.
3. Predict which of these operations will fail, then verify by running them in a try/except block:
   - `torch.ones(4, 3) + torch.ones(4, 3)` — will this work?
   - `torch.ones(4, 1) + torch.ones(1, 3)` — will this work?
   - `torch.ones(4, 3) + torch.ones(4, 2)` — will this work?

In [ ]:
torch.manual_seed(SEED)
features   = torch.randn(6, 5)
bias       = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
col_scale  = torch.tensor([[2.0], [3.0], [4.0]])    # (3, 1)
row_weights = torch.tensor([[1.0, 0.5, 0.25]])       # (1, 3)

# 1. Add bias to every row of features
biased = None  # YOUR CODE

# 2. Outer-product style multiply: col_scale * row_weights
scaled = None  # YOUR CODE
# What does each entry mean?
# YOUR COMMENT: ...

# 3. Predict and verify
ops = [
    ("(4,3)+(4,3)", lambda: torch.ones(4,3) + torch.ones(4,3)),
    ("(4,1)+(1,3)", lambda: torch.ones(4,1) + torch.ones(1,3)),
    ("(4,3)+(4,2)", lambda: torch.ones(4,3) + torch.ones(4,2)),
]
for name, op in ops:
    try:
        result = op()
        print(f"{name} -> OK, shape={result.shape}")
    except RuntimeError as e:
        print(f"{name} -> ERROR: {e}")

# ---- Verification ----
if biased is not None:
    print(f"\nbiased shape = {biased.shape}  (expected (6, 5))")
if scaled is not None:
    print(f"scaled shape = {scaled.shape}  (expected (3, 3))")
    print(f"scaled:\n{scaled}")

## Section 6 — Autograd and Differentiation

This is the section where PyTorch really diverges from NumPy. **Autograd** is the engine that makes backpropagation automatic.

**The key idea:** mark a tensor with `requires_grad=True` and PyTorch records every operation on it in a computation graph. When you call `.backward()` on a scalar loss, gradients flow backward through that graph via the chain rule.

```python
w = torch.tensor(3.0, requires_grad=True)
loss = w ** 2          # builds the graph: loss = w^2
loss.backward()        # d(loss)/d(w) = 2w = 6
print(w.grad)          # tensor(6.)
```

**Compared to TensorFlow's `tf.GradientTape`:**

```python
# TF2 way
with tf.GradientTape() as tape:
    loss = w ** 2
grad = tape.gradient(loss, w)

# PyTorch way
loss = w ** 2
loss.backward()
grad = w.grad          # cleaner — no context manager needed
```

**`torch.no_grad()`** — wrap inference code to skip building the graph and save memory:
```python
with torch.no_grad():
    predictions = model(x)   # no gradient tracking
```

### Demo — autograd

In [ ]:
# Demo 1: simple scalar gradient
w = torch.tensor(3.0, requires_grad=True)
loss = w ** 2    # loss = w^2, d/dw = 2w
loss.backward()
print(f"w = {w.item()},  loss = {loss.item()},  grad = {w.grad.item()}")
print(f"Expected grad: 2 * w = {2 * w.item()}")

# Demo 2: gradient of a multi-variable function
# f(x, y) = x^2 + 3*x*y   at x=2, y=5
# df/dx = 2x + 3y = 4 + 15 = 19
# df/dy = 3x          = 6
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(5.0, requires_grad=True)
f = x**2 + 3 * x * y
f.backward()
print(f"\nf(2,5) = {f.item()},  df/dx = {x.grad.item()},  df/dy = {y.grad.item()}")
print(f"Expected: df/dx = {2*x.item() + 3*y.item()},  df/dy = {3*x.item()}")

# Demo 3: torch.no_grad() disables tracking
with torch.no_grad():
    z = w ** 3           # no graph built
    print(f"\nz.requires_grad = {z.requires_grad}  (should be False)")

### Lab 6 — Autograd

**Tasks:**
1. Create a scalar tensor `a = 4.0` with `requires_grad=True`. Compute `loss_a = a ** 3`. Call `.backward()`. Store the gradient in `grad_a`. Verify it equals `3 * a^2 = 48`.
2. Create scalar tensors `p = 2.0` and `q = -3.0`, both with `requires_grad=True`. Compute `loss_pq = p * q + p ** 2`. Call `.backward()`. Store gradients in `grad_p` and `grad_q`. Verify analytically: `d/dp = q + 2p`, `d/dq = p`.
3. Compute `loss_pq` again but this time inside `torch.no_grad()`. Store it in `loss_no_grad`. Confirm that `loss_no_grad.requires_grad` is `False`.

In [ ]:
# 1. Gradient of a^3
a      = None  # YOUR CODE: tensor(4.0, requires_grad=True)
loss_a = None  # YOUR CODE: a ** 3
# YOUR CODE: call .backward()
grad_a = None  # YOUR CODE: a.grad

# 2. Multi-variable gradient
p      = None  # YOUR CODE: tensor(2.0, requires_grad=True)
q      = None  # YOUR CODE: tensor(-3.0, requires_grad=True)
loss_pq = None  # YOUR CODE: p * q + p ** 2
# YOUR CODE: call .backward()
grad_p = None  # YOUR CODE: p.grad
grad_q = None  # YOUR CODE: q.grad

# 3. No-grad block
loss_no_grad = None  # YOUR CODE: compute p*q + p**2 inside torch.no_grad()

# ---- Verification ----
if grad_a is not None:
    expected_a = 3 * (4.0 ** 2)
    print(f"grad_a = {grad_a.item():.1f}  expected = {expected_a:.1f}  OK={abs(grad_a.item()-expected_a)<1e-4}")
if grad_p is not None and grad_q is not None:
    p_val, q_val = 2.0, -3.0
    expected_p = q_val + 2 * p_val   # = -3 + 4 = 1
    expected_q = p_val                # = 2
    print(f"grad_p = {grad_p.item():.1f}  expected = {expected_p:.1f}  OK={abs(grad_p.item()-expected_p)<1e-4}")
    print(f"grad_q = {grad_q.item():.1f}  expected = {expected_q:.1f}  OK={abs(grad_q.item()-expected_q)<1e-4}")
if loss_no_grad is not None:
    print(f"loss_no_grad.requires_grad = {loss_no_grad.requires_grad}  (should be False)")

## Section 7 — Dataset and DataLoader

In TensorFlow you used `tf.data.Dataset` for batching and shuffling. PyTorch's equivalent is:

- `TensorDataset(X, y)` — wraps two (or more) tensors so they're indexed together.
- `DataLoader(dataset, batch_size=..., shuffle=True)` — iterates over mini-batches.

The pattern:

```python
# 1. Wrap tensors
ds = TensorDataset(X_tensor, y_tensor)

# 2. Build a loader
loader = DataLoader(ds, batch_size=32, shuffle=True)

# 3. Iterate
for x_batch, y_batch in loader:
    ...
```

Key parameters:
- `batch_size` — number of samples per mini-batch.
- `shuffle=True` — reshuffles before each epoch (critical for training, turn off for evaluation).
- `num_workers=0` — number of parallel data-loading processes (0 = main process; safe on Colab).
- `drop_last=False` — whether to discard the last incomplete batch.

### Demo — TensorDataset + DataLoader

In [ ]:
torch.manual_seed(SEED)

# Synthetic dataset: 100 samples, 8 features, binary labels
N, F = 100, 8
X_demo = torch.randn(N, F)                        # features
y_demo = (X_demo[:, 0] > 0).long()               # label: 1 if first feature > 0

# Wrap in TensorDataset
ds_demo = TensorDataset(X_demo, y_demo)
print(f"Dataset length: {len(ds_demo)}")
print(f"First sample: X shape={ds_demo[0][0].shape}, y={ds_demo[0][1].item()}")

# Build a DataLoader
loader_demo = DataLoader(ds_demo, batch_size=16, shuffle=True, num_workers=0)
print(f"\nNumber of batches per epoch: {len(loader_demo)}")

# Peek at the first batch
x_batch, y_batch = next(iter(loader_demo))
print(f"Batch X shape: {x_batch.shape}   Batch y shape: {y_batch.shape}")

### Lab 7 — Dataset and DataLoader

**Tasks:**
1. Create a synthetic regression dataset: `X_lab` of shape `(200, 4)` (random normal), and `y_lab` of shape `(200,)` where each label is `X_lab[:, 0] * 2 + X_lab[:, 1] - 1` (i.e., a linear function of the first two features). Use `dtype=torch.float32` for both.
2. Wrap them in a `TensorDataset` named `lab_dataset`.
3. Create a `DataLoader` named `lab_loader` with `batch_size=32`, `shuffle=True`, `num_workers=0`.
4. Iterate over the loader for one epoch. In each batch, compute the batch mean of `y` and print it. Count total samples processed; verify it equals 200.

In [ ]:
torch.manual_seed(SEED)

# 1. Create tensors
X_lab = None  # YOUR CODE: shape (200, 4), random normal, float32
y_lab = None  # YOUR CODE: X_lab[:,0]*2 + X_lab[:,1] - 1, float32

# 2. TensorDataset
lab_dataset = None  # YOUR CODE

# 3. DataLoader
lab_loader  = None  # YOUR CODE: batch_size=32, shuffle=True, num_workers=0

# 4. One-epoch iteration
total_samples = 0
for i, (xb, yb) in enumerate(lab_loader if lab_loader is not None else []):
    batch_mean_y = yb.mean().item()
    total_samples += xb.shape[0]
    print(f"  Batch {i:2d}: size={xb.shape[0]}, y_mean={batch_mean_y:.4f}")

# ---- Verification ----
if lab_dataset is not None:
    print(f"\nDataset length : {len(lab_dataset)}  (expected 200)")
if lab_loader is not None:
    print(f"Num batches    : {len(lab_loader)}   (expected 7 for batch_size=32, drop_last=False)")
if total_samples:
    print(f"Total samples  : {total_samples}  (expected 200)")

## Section 8 — Simple Neural Network Components

Everything you've built so far converges here. A neural network in PyTorch is:

1. A class that inherits from `nn.Module`.
2. Layers defined in `__init__`.
3. The forward pass wired together in `forward`.

**The minimal two-layer network:**

```python
class TinyNet(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)    # weights + bias
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden, out_features)

    def forward(self, x):
        x = self.fc1(x)     # (B, in) -> (B, hidden)
        x = self.act(x)     # element-wise ReLU
        x = self.fc2(x)     # (B, hidden) -> (B, out)
        return x            # logits — no softmax!
```

**The six-line training loop:**

```python
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn   = nn.CrossEntropyLoss()   # or nn.MSELoss for regression

for x, y in loader:
    optimizer.zero_grad()    # 1. clear stale gradients
    logits = model(x)        # 2. forward
    loss   = loss_fn(logits, y)  # 3. loss
    loss.backward()          # 4. backprop
    optimizer.step()         # 5. update weights
```

### Demo — forward pass shape check

In [ ]:
torch.manual_seed(SEED)

# Demo model: 4 inputs -> 8 hidden -> 2 outputs (binary classification)
demo_net = nn.Sequential(
    nn.Linear(4, 8),    # weight: (8, 4), bias: (8,)
    nn.ReLU(),
    nn.Linear(8, 2),    # weight: (2, 8), bias: (2,)
)

print("Model architecture:")
print(demo_net)

total_params = sum(p.numel() for p in demo_net.parameters())
print(f"\nTotal trainable parameters: {total_params}")

# Single forward pass — batch of 5 samples
x_dummy = torch.randn(5, 4)                  # (batch=5, features=4)
logits_demo = demo_net(x_dummy)
print(f"\nInput  shape: {x_dummy.shape}")
print(f"Logits shape: {logits_demo.shape}  (expected (5, 2))")

# One backward pass to verify gradients flow
loss_demo = logits_demo.mean()               # dummy scalar loss
loss_demo.backward()
print(f"Gradient on first linear weight: {demo_net[0].weight.grad.shape}")

### Lab 8 — Build and Train a Tiny Network

This is the capstone lab — it uses everything from sections 1–7.

**Tasks:**
1. Define class `SmallNet(nn.Module)` with:
   - `fc1`: `nn.Linear(4, 16)` 
   - `relu`: `nn.ReLU()`
   - `fc2`: `nn.Linear(16, 1)` (single output for regression)
   - `forward(x)`: passes `x` through `fc1 -> relu -> fc2`, returns the output.
2. Instantiate `model_lab = SmallNet().to(device)`.
3. Use the `lab_dataset` and `lab_loader` you built in Lab 7.
4. Define `loss_fn = nn.MSELoss()` and `optimizer = torch.optim.Adam(model_lab.parameters(), lr=1e-2)`.
5. Write the training loop for 10 epochs. In each epoch:
   - Iterate over `lab_loader`.
   - Run the six-line inner loop (zero_grad → forward → loss → backward → step).
   - Track average loss per epoch and print it.
6. After training, run one inference batch with `torch.no_grad()` and print the first 5 predictions vs ground-truth labels.

In [ ]:
torch.manual_seed(SEED)

# 1. Define SmallNet
class SmallNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1  = None  # YOUR CODE: nn.Linear(4, 16)
        self.relu = None  # YOUR CODE: nn.ReLU()
        self.fc2  = None  # YOUR CODE: nn.Linear(16, 1)

    def forward(self, x):
        # YOUR CODE: fc1 -> relu -> fc2
        return None  # YOUR CODE

# 2. Instantiate
model_lab = None  # YOUR CODE: SmallNet().to(device)

# 3. Loss and optimizer
loss_fn   = None  # YOUR CODE: nn.MSELoss()
optimizer = None  # YOUR CODE: torch.optim.Adam(model_lab.parameters(), lr=1e-2)

# 5. Training loop — 10 epochs
EPOCHS_LAB = 10
epoch_losses_lab = []

for epoch in range(EPOCHS_LAB):
    if model_lab is None or lab_loader is None:
        break
    model_lab.train()
    total_loss, n_batches = 0.0, 0
    for xb, yb in lab_loader:
        xb = xb.to(device)
        yb = yb.to(device).unsqueeze(1)   # (B,) -> (B, 1) to match fc2 output

        # YOUR CODE: zero_grad
        # YOUR CODE: forward pass  ->  preds
        preds = None  # YOUR CODE
        # YOUR CODE: loss
        loss  = None  # YOUR CODE
        # YOUR CODE: backward
        # YOUR CODE: optimizer step

        if loss is not None:
            total_loss += loss.item()
        n_batches += 1

    avg = total_loss / max(n_batches, 1)
    epoch_losses_lab.append(avg)
    print(f"Epoch {epoch+1:2d}/{EPOCHS_LAB}  avg MSE loss = {avg:.4f}")

# 6. Inference on the first batch
if model_lab is not None and lab_loader is not None:
    model_lab.eval()
    xb_infer, yb_infer = next(iter(lab_loader))
    xb_infer = xb_infer.to(device)
    with torch.no_grad():
        preds_infer = None  # YOUR CODE: run model_lab on xb_infer, .squeeze(1)
    if preds_infer is not None:
        print("\nFirst 5 predictions vs ground truth:")
        for pred, true in zip(preds_infer[:5].cpu(), yb_infer[:5]):
            print(f"  pred={pred.item():.4f}   true={true.item():.4f}")

## Optional Lab — Gradient Descent from Scratch

Implement linear regression **without** `nn.Linear` — just raw tensors and autograd. This gives you the deepest intuition for what `optimizer.step()` is actually doing.

**Setup:** `y = 2*x + 1 + noise`. Learn `w` and `b` such that `y_pred = w*x + b`.

**Steps:**
1. Generate 50 data points: `x = torch.linspace(-1, 1, 50)`, `y = 2*x + 1 + 0.1*randn(50)`.
2. Initialize `w = torch.tensor(0.0, requires_grad=True)` and `b = torch.tensor(0.0, requires_grad=True)`.
3. For 200 gradient descent steps with `lr = 0.1`:
   - Compute `y_pred = w * x + b`
   - Compute MSE loss: `loss = ((y_pred - y) ** 2).mean()`
   - Call `loss.backward()`
   - Update `w` and `b` inside `torch.no_grad()`: `w -= lr * w.grad; b -= lr * b.grad`
   - Zero the gradients manually: `w.grad.zero_(); b.grad.zero_()`
4. Print the final `w` and `b` — they should be close to `2.0` and `1.0`.

In [ ]:
torch.manual_seed(SEED)

# Optional: gradient descent from scratch
lr = 0.1
STEPS = 200

# 1. Data
x_gd = torch.linspace(-1, 1, 50)
y_gd = 2 * x_gd + 1 + 0.1 * torch.randn(50)

# 2. Parameters (YOUR CODE)
w_gd = None  # YOUR CODE: tensor(0.0, requires_grad=True)
b_gd = None  # YOUR CODE: tensor(0.0, requires_grad=True)

# 3. Training loop (YOUR CODE)
for step in range(STEPS):
    y_pred_gd = None  # YOUR CODE
    loss_gd   = None  # YOUR CODE: MSE
    # YOUR CODE: backward
    with torch.no_grad():
        pass  # YOUR CODE: w_gd -= lr * w_gd.grad; b_gd -= lr * b_gd.grad
    # YOUR CODE: zero grads

# 4. Print results
if w_gd is not None:
    print(f"Learned w = {w_gd.item():.4f}  (expected ~2.0)")
    print(f"Learned b = {b_gd.item():.4f}  (expected ~1.0)")

## Congratulations!

You have completed the PyTorch Tensor Exercises. Here is what you built and learned:

### Key takeaways

| Section | Core skill |
|---------|-----------|
| 1 — Tensor Basics | `torch.tensor`, factory functions, `.shape`, `.dtype`, numpy ↔ torch |
| 2 — Operations | Arithmetic, `mean/std`, `relu`, `@` matrix multiply |
| 3 — Shape Manipulation | `reshape/view`, `transpose/permute`, `flatten`, `squeeze/unsqueeze` |
| 4 — Indexing | Row/col access, slicing, boolean masking |
| 5 — Broadcasting | Shape alignment rules, bias-addition pattern |
| 6 — Autograd | `requires_grad`, `.backward()`, `.grad`, `torch.no_grad()` |
| 7 — DataLoader | `TensorDataset`, `DataLoader`, batching + shuffling |
| 8 — nn.Module | `nn.Linear`, `nn.ReLU`, the 6-line training loop |

### Self-check questions

1. What is the difference between `.reshape()` and `.view()`? When does `.view()` fail?
2. If `A.shape = (3, 1)` and `B.shape = (1, 4)`, what is `(A + B).shape` and why?
3. What happens if you forget `optimizer.zero_grad()` inside the training loop?
4. Why should you **not** apply `softmax` in `forward()` when using `nn.CrossEntropyLoss`?

### Next steps

- **Notebook 5 — CBOW Word Embeddings**: uses `nn.Embedding`, `Dataset`, and the full training loop you just practiced.
- **Notebook 8 — MLP Text Classification**: builds a multi-layer network on 130K news headlines.
- PyTorch official tutorials: https://pytorch.org/tutorials/